1. environment set up


In [ ]:
!pip install -q ollama-python==0.1.2
!pip install -q ddgs==9.4.0
!pip install -q langchain==0.3.26
!pip install -q langchain-openai==0.3.27
!pip install -q langchain-community==0.3.27
!pip install -q openai==1.86.0
!pip install -q fastapi==0.116.1
!pip install -q uvicorn==0.35.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires httpx<1,>=0.27, but you have httpx 0.26.0 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.26.0 which is incompatible.
google-genai 2.11.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.26.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.6 M

In [ ]:
!pip install -q ollama

In [ ]:
!pip uninstall -y ollama-python

Found existing installation: ollama-python 0.1.2
Uninstalling ollama-python-0.1.2:
  Successfully uninstalled ollama-python-0.1.2


In [ ]:
import langchain
import openai
import fastapi
from ddgs import DDGS
import ollama

print("langchain:", langchain.__version__)
print("openai:", openai.__version__)
print("fastapi:", fastapi.__version__)
print("All imports successful")

langchain: 0.3.26
openai: 1.86.0
fastapi: 0.116.1
All imports successful


In [ ]:
# Web search — this is what makes it a "web agent"
from ddgs import DDGS
# Ollama
import ollama
# OpenAI
from openai import OpenAI
# LangChain core pieces
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate
# LangChain's OpenAI + Community integrations
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.llms import Ollama
from langchain_community.chat_models import ChatOllama
# For building a small API around your agent
from fastapi import FastAPI
import uvicorn

ollama setup


In [ ]:
# 1. Install zstd + Ollama
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (511 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current us

In [ ]:
# Start Ollama server in background
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

In [ ]:
!ollama pull gemma3:12b

2.tool calling


In [ ]:
# Step 1: Implement the tool

def get_current_weather(city, unit ="celsius"):
    return f"It is 23 {unit[0].upper()} and sunny in {city}"

get_current_weather("London")

'It is 23 C and sunny in London'

In [ ]:
# Step 2: Create the prompt for the LLM to call tool
# Goal:
#   Build the system and user prompts that instruct the model when and how
#   to use your tool (`get_current_weather`).

SYSTEM_PROMPT = (
    "You are an assistant that can call tools."
    "When the user asks something requirig current data , respond **only** with JSON like:"
    'TOOL_CALL:{"name":<tool_name>,"args":{...}}.'
)

TOOL_SPECS = """
you can call one tool:
-name: get_current_weather
-description: get the current weather for a city
arguments:
 city:string
 units: "celsius" or "fahrenheit"
"""

user_question = "what is weather in bangalore today?"

In [ ]:
# Step 3: Call the LLM with your prompt
from openai import OpenAI
client = OpenAI(api_key = "ollama", base_url = "http://localhost:11434/v1")

response = client.chat.completions.create(
    model = "gemma3:12b",
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_question},
    ],
    temperature=0.3
)
raw = response.choices[0].message.content
print("\n model output:\n",raw)



 model output:
 TOOL_CALL:{"name":"weather","args":{"location":"bangalore"}}


In [ ]:
import json

decoder = json.JSONDecoder()
call_json, _ = decoder.raw_decode(match.group(1))
print(call_json, type(call_json))

NameError: name 'match' is not defined

In [ ]:
# Step 4: Parse the LLM output and call the tool
import re, json

match = re.search(pattern, raw, flags=re.DOTALL)

decoder = json.JSONDecoder()
call_json, _ = decoder.raw_decode(match.group(1))
print(call_json, type(call_json))

NameError: name 'pattern' is not defined

In [ ]:
def get_current_weather(city: str, unit: str = "celsius"):
  "geting the current weather for a given city."
  return f"It is 23 {unit[0].upper()} and sunny in {city}"

get_current_weather("bangalore")

'It is 23 C and sunny in bangalore'

3. standardize tool calling


In [ ]:
from os.path import split
from os import name
# Generate a JSON schema for a tool automatically
from pprint import pprint
import inspect

def to_schema(fn):
  sig = inspect.signature(fn)
  props = {
      name: {
          'type': 'string' if param.annotation is str else 'number',
          'description': f"Argument {name}"
      }
      for name , param in sig.parameters.items()
  }
  tool_schema = {
      "name": fn.__name__,
      "description": (fn.__doc__ or '').strip().split("\n")[0],
      "parameters": {
          "type": 'object',
          "properties": props,
          "required": [n for n , p in sig.parameters.items() if p.default is inspect._empty]
      }
  }
  return tool_schema
fns = [get_current_weather]
tool_schema = []
for tool_fn in fns:
  tool_schema.append(to_schema(tool_fn))
pprint(tool_schema)


[{'description': '',
  'name': 'get_current_weather',
  'parameters': {'properties': {'city': {'description': 'Argument city',
                                         'type': 'number'},
                                'unit': {'description': 'Argument unit',
                                         'type': 'number'}},
                 'required': ['city'],
                 'type': 'object'}}]


In [ ]:
# Provide the tool schema to the model
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "system","content": json.dumps(tool_schema)},
    {"role": "user", "content": user_question},
]
model= "gemma3:12b"
raw = client.chat.completions.create(
    model = model,
    messages = messages,
    temperature=0.3
)
print(raw.choices[0].message.content)

TOOL_CALL:{"name":"get_current_weather","args":{"city":bangalore,"unit":1}}


4. Langchain for tool calling


In [ ]:
#Step 1: Define tools for LangChain

from langchain.tools import tool

@tool
def get_current_weather(city: str, unit: str = "celsius") -> str:
  "geting the current weather for a given city."
  return f"It is 23 {unit[0].upper()} and sunny in {city}"

def get_weather(city: str) -> str:
    return get_current_weather(city)

In [ ]:
# Step 2: Initialize the LangChain Agent

from langchain_community.chat_models import ChatOllama
from langchain.agents import initialize_agent, AgentType, Tool

Model = "gemma3:12b"
llm = ChatOllama(model=Model, temperature=0.3)
tools = [get_current_weather]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)
agent.invoke({"input": "do i need umbrella today in tiptur"})




> Entering new AgentExecutor chain...
Thought: I need to check the weather in Tiptur to determine if an umbrella is needed.
Action:
```json
{
  "action": "get_current_weather",
  "action_input": {
    "city": "Tiptur",
    "unit": "celsius"
  }
}
```
Observation: It is 23 C and sunny in Tiptur
Thought:Action:
```json
{
  "action": "Final Answer",
  "action_input": "No, you don't need an umbrella today in Tiptur. The weather is currently 23°C and sunny."
}
```

> Finished chain.


{'input': 'do i need umbrella today in tiptur',
 'output': "No, you don't need an umbrella today in Tiptur. The weather is currently 23°C and sunny."}

5. Perplexity style web search

In [ ]:
# Step 1: Add a web search tool
from ddgs import DDGS
from langchain.tools import tool

def _search_web_impl(query: str, max_results: int = 10) -> str:
    """Return the top 'max_results' web results for 'query' as a single formatted string. Uses DuckDuckGo's API."""
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=max_results):
            results.append(f"{r['title']}\n{r['href']}")
    return "\n".join(results)

@tool
def search_web(query: str) -> str:
    """Search the web for 'query' and return concise results."""
    return _search_web_impl(query)


In [ ]:

# Step 2: Initialize the web-search agent

from langchain.agents import initialize_agent, AgentType
from langchain_community.chat_models import ChatOllama

MODEL = "gemma3:12b"
llm = ChatOllama(model=MODEL, temperature=0.3)
tools = [search_web]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

In [ ]:
# Step 3: Test your Ask-the-Web agent

agent.invoke({"input": "is tiptur warmer today or bangalore"})




> Entering new AgentExecutor chain...
I need to compare the temperatures in Tiptur and Bangalore. To do this, I will search for the current weather conditions in both cities.
Action: search_web
Action Input: "current temperature in Tiptur"
Observation: Tiptur, KA, IN Current Weather - The Weather Network

Tiptur, Karnataka, India Current Weather | AccuWeather
https://www.accuweather.com/en/in/tiptur/193444/current-weather/193444
Tiptur Weather Tiptur Weather forecast today & tomorrow Current...
https://www.meteoprog.com/weather/Tiptur/
Weather inTiptur - accurate and detailed weather forecast inTiptur...
https://meteotrend.com/forecast/in/tiptur/
Today's Weather inTiptūr - Hourly Forecast and Conditions
https://www.easeweather.com/asia/india/karnataka/tumkur/tiptur/today
Weather inTiptūr, India - Finnish Meteorological Institute
https://en.ilmatieteenlaitos.fi/weather/india/tiptūr
Tiptur, India 10-Day Weather Forecast | Weather Underground
https://www.wunderground.com/forecast/in/tip

{'input': 'is tiptur warmer today or bangalore', 'output': 'Tiptur'}